# 🏦 Advanced Loan Default Risk Assessment

A comprehensive ML pipeline for credit risk scoring using the German Credit dataset (OpenML). Covers the full lifecycle: data loading, preprocessing, SMOTE balancing, 5-model training, SHAP explainability, business cost optimisation, 4-tier risk scorecard, calibration/learning curves, 16 professional plots, and production export.

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/jameskoero/loan-risk-assessment/blob/main/loan_risk_assessment.ipynb)

In [ ]:
!pip install imbalanced-learn xgboost lightgbm shap -q

In [ ]:
import matplotlib
matplotlib.use('Agg')
import os, json, warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import joblib, shap

from sklearn.datasets import fetch_openml
from sklearn.model_selection import train_test_split, learning_curve, StratifiedKFold
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.compose import ColumnTransformer
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.metrics import (accuracy_score, roc_auc_score, f1_score,
    precision_score, recall_score, confusion_matrix, roc_curve)
from sklearn.calibration import CalibrationDisplay
from imblearn.over_sampling import SMOTE
from xgboost import XGBClassifier
from lightgbm import LGBMClassifier

warnings.filterwarnings('ignore')
sns.set_theme(style='whitegrid', palette='muted')
PLOTS_DIR = 'plots'
os.makedirs(PLOTS_DIR, exist_ok=True)
colors = ['steelblue','tomato','seagreen','darkorange','mediumpurple']
print("Setup complete.")

## 1 · Data Loading
Auto-downloads the **German Credit (credit-g)** dataset from OpenML via scikit-learn. No separate openml package required.

In [ ]:
data = fetch_openml(name='credit-g', version=1, as_frame=True)
X = data.frame.drop(columns=['class'])
y = (data.frame['class'] == 'bad').astype(int)
print(f"Shape: {X.shape}")
print(f"Target distribution:\n{y.value_counts()}")
X.head()

In [ ]:
print(X.dtypes.value_counts())
print(X.isnull().sum().sum(), "missing values")
X.describe()

In [ ]:
fig, ax = plt.subplots(figsize=(6,4))
counts = y.value_counts()
ax.bar(['Good (0)', 'Bad (1)'], counts.values, color=['steelblue','tomato'])
ax.set_title('Class Distribution')
ax.set_ylabel('Count')
for i,v in enumerate(counts.values): ax.text(i, v+5, str(v), ha='center', fontweight='bold')
plt.tight_layout(); fig.savefig(f'{PLOTS_DIR}/01_class_distribution.png', dpi=150); plt.show()

## 2 · Preprocessing
LabelEncoder for categoricals, StandardScaler for numerics, combined via ColumnTransformer.

In [ ]:
categorical_cols = X.select_dtypes(include=['object','category']).columns.tolist()
numeric_cols     = X.select_dtypes(include=[np.number]).columns.tolist()

label_encoders = {}
X_encoded = X.copy()
for col in categorical_cols:
    le = LabelEncoder()
    X_encoded[col] = le.fit_transform(X[col].astype(str))
    label_encoders[col] = le

preprocessor = ColumnTransformer(transformers=[
    ('num', StandardScaler(), numeric_cols),
    ('cat', 'passthrough', categorical_cols),
])
X_proc = preprocessor.fit_transform(X_encoded)
feature_names = numeric_cols + categorical_cols
print(f"Processed shape: {X_proc.shape}")

In [ ]:
df_corr = pd.DataFrame(X_proc, columns=feature_names)
fig, ax = plt.subplots(figsize=(14,10))
mask = np.triu(np.ones_like(df_corr.corr(), dtype=bool))
sns.heatmap(df_corr.corr(), mask=mask, annot=False, cmap='coolwarm', center=0, ax=ax)
ax.set_title('Feature Correlation Heatmap')
plt.tight_layout(); fig.savefig(f'{PLOTS_DIR}/02_correlation_heatmap.png', dpi=150); plt.show()

## 3 · Train / Test Split & SMOTE

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X_proc, y, test_size=0.2, random_state=42, stratify=y)
print(f"Train: {X_train.shape}, Test: {X_test.shape}")

In [ ]:
sm = SMOTE(random_state=42)
X_train_res, y_train_res = sm.fit_resample(X_train, y_train)
print(f"After SMOTE — {pd.Series(y_train_res).value_counts().to_dict()}")

## 4 · Model Training
Five classifiers trained on the SMOTE-balanced training set.

In [ ]:
models = {
    'Logistic Regression': LogisticRegression(max_iter=1000, random_state=42),
    'Random Forest':       RandomForestClassifier(n_estimators=200, random_state=42),
    'Gradient Boosting':   GradientBoostingClassifier(n_estimators=200, random_state=42),
    'XGBoost':             XGBClassifier(n_estimators=200,
                               eval_metric='logloss', random_state=42, verbosity=0),
    'LightGBM':            LGBMClassifier(n_estimators=200, random_state=42, verbose=-1),
}

In [ ]:
fitted_models = {}
for name, model in models.items():
    model.fit(X_train_res, y_train_res)
    fitted_models[name] = model
    print(f"  ✓ {name}")

## 5 · Unified Evaluation

In [ ]:
def evaluate(name, model, X_test, y_test, threshold=0.5):
    proba = model.predict_proba(X_test)[:,1]
    preds = (proba >= threshold).astype(int)
    return dict(model=name,
                accuracy=accuracy_score(y_test, preds),
                roc_auc=roc_auc_score(y_test, proba),
                f1=f1_score(y_test, preds),
                precision=precision_score(y_test, preds),
                recall=recall_score(y_test, preds),
                confusion_matrix=confusion_matrix(y_test, preds),
                proba=proba)

results = [evaluate(n, m, X_test, y_test) for n, m in fitted_models.items()]
results_df = pd.DataFrame([{k:v for k,v in r.items() if k not in ('confusion_matrix','proba')}
                            for r in results])
print(results_df.to_string(index=False))

In [ ]:
fig, ax = plt.subplots(figsize=(8,6))
for res, color in zip(results, colors):
    fpr, tpr, _ = roc_curve(y_test, res['proba'])
    ax.plot(fpr, tpr, color=color, lw=2, label=f"{res['model']} (AUC={res['roc_auc']:.3f})")
ax.plot([0,1],[0,1],'k--',lw=1)
ax.set(xlabel='FPR', ylabel='TPR', title='ROC Curves — All Models')
ax.legend(loc='lower right')
plt.tight_layout(); fig.savefig(f'{PLOTS_DIR}/03_roc_curves.png', dpi=150); plt.show()

In [ ]:
df_melt = results_df.melt(id_vars='model', var_name='Metric', value_name='Score')
fig, ax = plt.subplots(figsize=(12,6))
sns.barplot(data=df_melt, x='model', y='Score', hue='Metric', ax=ax)
ax.set(title='Model Performance Comparison', ylim=(0,1), xlabel='')
ax.tick_params(axis='x', rotation=15)
ax.legend(bbox_to_anchor=(1.01,1), loc='upper left')
plt.tight_layout(); fig.savefig(f'{PLOTS_DIR}/04_model_comparison.png', dpi=150); plt.show()

In [ ]:
n = len(results)
fig, axes = plt.subplots(1, n, figsize=(4*n, 4))
for ax, res in zip(axes, results):
    sns.heatmap(res['confusion_matrix'], annot=True, fmt='d', cmap='Blues', ax=ax, cbar=False)
    ax.set(title=res['model'], xlabel='Predicted', ylabel='Actual')
plt.suptitle('Confusion Matrices', fontsize=14, y=1.02)
plt.tight_layout(); fig.savefig(f'{PLOTS_DIR}/05_confusion_matrices.png', dpi=150); plt.show()

## 6 · Best Model & Feature Importance

In [ ]:
best_result = max(results, key=lambda r: r['roc_auc'])
best_name   = best_result['model']
best_model  = fitted_models[best_name]
best_proba  = best_result['proba']
print(f"Best model: {best_name}  AUC={best_result['roc_auc']:.4f}")

In [ ]:
importances = best_model.feature_importances_
idx = np.argsort(importances)[-20:]
fig, ax = plt.subplots(figsize=(8,7))
ax.barh(np.array(feature_names)[idx], importances[idx], color='steelblue')
ax.set(title=f'Top 20 Feature Importances — {best_name}', xlabel='Importance')
plt.tight_layout(); fig.savefig(f'{PLOTS_DIR}/06_feature_importance.png', dpi=150); plt.show()

## 7 · Precision-Recall & Probability Distributions

In [ ]:
from sklearn.metrics import precision_recall_curve, average_precision_score
fig, ax = plt.subplots(figsize=(8,6))
for res, color in zip(results, colors):
    prec, rec, _ = precision_recall_curve(y_test, res['proba'])
    ap = average_precision_score(y_test, res['proba'])
    ax.plot(rec, prec, color=color, lw=2, label=f"{res['model']} (AP={ap:.3f})")
ax.set(xlabel='Recall', ylabel='Precision', title='Precision-Recall Curves')
ax.legend(loc='upper right')
plt.tight_layout(); fig.savefig(f'{PLOTS_DIR}/07_precision_recall_curves.png', dpi=150); plt.show()

In [ ]:
fig, axes = plt.subplots(1, n, figsize=(4*n, 4), sharey=True)
for ax, res in zip(axes, results):
    for label, color in [(0,'steelblue'),(1,'tomato')]:
        mask = y_test == label
        ax.hist(res['proba'][mask], bins=20, alpha=0.6, color=color,
                label=f'Class {label}', density=True)
    ax.set(title=res['model'], xlabel='P(default)')
    ax.legend(fontsize=8)
axes[0].set_ylabel('Density')
plt.suptitle('Probability Distributions', fontsize=14, y=1.02)
plt.tight_layout(); fig.savefig(f'{PLOTS_DIR}/08_probability_distributions.png', dpi=150); plt.show()

## 8 · Business Cost Matrix & Optimal Threshold
FN cost = 5× FP cost. Search over 200 thresholds for the minimum total cost.

In [ ]:
FN_COST, FP_COST = 5, 1
thresholds = np.linspace(0.01, 0.99, 200)
best_thresh, best_cost = 0.5, np.inf
costs = []
for t in thresholds:
    preds = (best_proba >= t).astype(int)
    tn, fp, fn, tp = confusion_matrix(y_test, preds).ravel()
    c = FN_COST*fn + FP_COST*fp
    costs.append(c)
    if c < best_cost: best_cost, best_thresh = c, t

fig, ax = plt.subplots(figsize=(8,5))
ax.plot(thresholds, costs, color='steelblue', lw=2)
ax.axvline(best_thresh, color='red', linestyle='--', label=f'Optimal θ={best_thresh:.2f}')
ax.set(xlabel='Threshold', ylabel='Total Business Cost',
       title=f'Cost vs. Threshold — {best_name}')
ax.legend()
plt.tight_layout(); fig.savefig(f'{PLOTS_DIR}/09_cost_threshold.png', dpi=150); plt.show()
print(f"Optimal threshold: {best_thresh:.3f}  Cost: {best_cost}")

## 9 · 4-Tier Risk Scorecard

In [ ]:
tiers = pd.cut(best_proba, bins=[-np.inf,0.25,0.50,0.75,np.inf],
               labels=['Low','Medium','High','Very High'])
df_sc = pd.DataFrame({'tier': tiers, 'actual': y_test.values})
summary = df_sc.groupby('tier', observed=True)['actual'].agg(count='count', default_rate='mean')
print(summary)

tier_colors = ['seagreen','gold','darkorange','tomato']
fig, axes = plt.subplots(1,2, figsize=(12,5))
axes[0].bar(summary.index, summary['count'], color=tier_colors)
axes[0].set(title='Loan Count by Risk Tier', xlabel='Risk Tier', ylabel='Count')
axes[1].bar(summary.index, summary['default_rate'], color=tier_colors)
axes[1].set(title='Default Rate by Risk Tier', xlabel='Risk Tier', ylabel='Default Rate', ylim=(0,1))
plt.tight_layout(); fig.savefig(f'{PLOTS_DIR}/10_risk_scorecard.png', dpi=150); plt.show()

## 10 · Calibration Curves

In [ ]:
fig, ax = plt.subplots(figsize=(8,6))
for (name, model), color in zip(fitted_models.items(), colors):
    CalibrationDisplay.from_estimator(model, X_test, y_test,
        n_bins=10, ax=ax, name=name, color=color)
ax.set_title('Calibration Curves — All Models')
plt.tight_layout(); fig.savefig(f'{PLOTS_DIR}/11_calibration_curves.png', dpi=150); plt.show()

## 11 · Learning Curves

In [ ]:
train_sizes, train_scores, val_scores = learning_curve(
    best_model, X_train, y_train,
    cv=StratifiedKFold(n_splits=5, shuffle=True, random_state=42),
    scoring='roc_auc', train_sizes=np.linspace(0.1,1.0,10), n_jobs=-1)

fig, ax = plt.subplots(figsize=(8,5))
ax.plot(train_sizes, train_scores.mean(axis=1), 'o-', color='steelblue', label='Train AUC')
ax.fill_between(train_sizes,
    train_scores.mean(1)-train_scores.std(1),
    train_scores.mean(1)+train_scores.std(1), alpha=0.2, color='steelblue')
ax.plot(train_sizes, val_scores.mean(axis=1), 'o-', color='tomato', label='CV AUC')
ax.fill_between(train_sizes,
    val_scores.mean(1)-val_scores.std(1),
    val_scores.mean(1)+val_scores.std(1), alpha=0.2, color='tomato')
ax.set(title=f'Learning Curves — {best_name}', xlabel='Training Samples', ylabel='ROC-AUC')
ax.legend()
plt.tight_layout(); fig.savefig(f'{PLOTS_DIR}/12_learning_curves.png', dpi=150); plt.show()

## 12 · SHAP Explainability
TreeExplainer provides exact SHAP values for tree-based models in near-linear time.

In [ ]:
explainer = shap.TreeExplainer(best_model)
X_sample = X_test[:200]
shap_values = explainer.shap_values(X_sample)
if isinstance(shap_values, list): shap_values = shap_values[1]

mean_abs = np.abs(shap_values).mean(axis=0)
idx = np.argsort(mean_abs)[-20:]
fig, ax = plt.subplots(figsize=(8,7))
ax.barh(np.array(feature_names)[idx], mean_abs[idx], color='steelblue')
ax.set(title=f'SHAP Mean |Value| — {best_name}', xlabel='Mean |SHAP Value|')
plt.tight_layout(); fig.savefig(f'{PLOTS_DIR}/13_shap_bar.png', dpi=150); plt.show()

In [ ]:
explanation = shap.Explanation(values=shap_values, data=X_sample, feature_names=feature_names)
fig, ax = plt.subplots(figsize=(10,8))
shap.plots.beeswarm(explanation, max_display=15, show=False)
plt.title(f'SHAP Beeswarm — {best_name}')
plt.tight_layout(); fig.savefig(f'{PLOTS_DIR}/14_shap_beeswarm.png', dpi=150); plt.show()

In [ ]:
fig, ax = plt.subplots(figsize=(8,5))
ax.hist(best_proba[y_test==0], bins=30, alpha=0.6, color='steelblue', label='Good', density=True)
ax.hist(best_proba[y_test==1], bins=30, alpha=0.6, color='tomato',    label='Bad',  density=True)
ax.set(xlabel='Risk Score P(default)', ylabel='Density',
       title=f'Risk Score Distribution — {best_name}')
ax.legend()
plt.tight_layout(); fig.savefig(f'{PLOTS_DIR}/15_score_distribution.png', dpi=150); plt.show()

In [ ]:
accs, f1s, precs, recs = [], [], [], []
for t in thresholds:
    preds = (best_proba >= t).astype(int)
    accs.append(accuracy_score(y_test, preds))
    f1s.append(f1_score(y_test, preds, zero_division=0))
    precs.append(precision_score(y_test, preds, zero_division=0))
    recs.append(recall_score(y_test, preds, zero_division=0))

fig, ax = plt.subplots(figsize=(9,5))
ax.plot(thresholds, accs,  label='Accuracy', lw=2)
ax.plot(thresholds, f1s,   label='F1',       lw=2)
ax.plot(thresholds, precs, label='Precision',lw=2)
ax.plot(thresholds, recs,  label='Recall',   lw=2)
ax.axvline(best_thresh, color='red', linestyle='--', label=f'Optimal θ={best_thresh:.2f}')
ax.set(xlabel='Threshold', ylabel='Score',
       title=f'Metrics vs. Threshold — {best_name}')
ax.legend()
plt.tight_layout(); fig.savefig(f'{PLOTS_DIR}/16_threshold_metrics.png', dpi=150); plt.show()
print("All 16 plots saved.")

## 14 · Production Export

In [ ]:
joblib.dump(best_model, 'loan_risk_model.joblib')
metadata = dict(
    model_name=best_name,
    roc_auc=round(best_result['roc_auc'],4),
    accuracy=round(best_result['accuracy'],4),
    f1=round(best_result['f1'],4),
    precision=round(best_result['precision'],4),
    recall=round(best_result['recall'],4),
    fn_cost_multiplier=FN_COST,
    fp_cost=FP_COST,
)
with open('model_metadata.json','w') as f: json.dump(metadata, f, indent=2)
print("Exported: loan_risk_model.joblib")
print("Metadata:", json.dumps(metadata, indent=2))